In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 1 – IMPORTS
# ─────────────────────────────────────────────────────────────────
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 2 – LOAD DATASET
# ─────────────────────────────────────────────────────────────────
df_raw = pd.read_csv('house_prices.csv')
print(f'Dataset loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
display(df_raw.head())

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 3 – DATA CLEANING
# ─────────────────────────────────────────────────────────────────

df = df_raw.copy()

# --- 3a. Parse 'Amount(in rupees)' → numeric rupee value --------
def parse_amount(val):
    """Convert strings like '42 Lac', '1.40 Cr', 'Call for Price' → float (INR)."""
    if pd.isna(val):
        return np.nan
    val = str(val).strip().lower()
    if 'call' in val or val == '':
        return np.nan
    val = val.replace(',', '')
    num_match = re.search(r'[\d.]+', val)
    if not num_match:
        return np.nan
    num = float(num_match.group())
    if 'cr' in val:
        return num * 1e7
    if 'lac' in val or 'lakh' in val:
        return num * 1e5
    return num

df['price_inr'] = df['Amount(in rupees)'].apply(parse_amount)

# --- 3b. Parse 'Carpet Area' / 'Super Area' → numeric sqft ------
def parse_area(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip().lower().replace(',', '')
    m = re.search(r'[\d.]+', val)
    return float(m.group()) if m else np.nan

df['carpet_area_sqft'] = df['Carpet Area'].apply(parse_area)
df['super_area_sqft']  = df['Super Area'].apply(parse_area)

# --- 3c. Derive area: prefer carpet_area, fall back to super_area
df['area_sqft'] = df['carpet_area_sqft'].combine_first(df['super_area_sqft'])

# --- 3d. Extract BHK count from Title --------------------------
def extract_bhk(title):
    m = re.search(r'(\d+)\s*BHK', str(title), re.IGNORECASE)
    return int(m.group(1)) if m else np.nan

df['bhk'] = df['Title'].apply(extract_bhk)

# --- 3e. Parse Floor (current floor) ---------------------------
def parse_floor(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip().lower()
    if 'ground' in val:
        return 0
    m = re.search(r'(\d+)\s*out', val)
    return int(m.group(1)) if m else np.nan

df['floor_num'] = df['Floor'].apply(parse_floor)

# --- 3f. Encode categorical columns ---------------------------
cat_cols = ['Furnishing', 'Status', 'Transaction', 'facing', 'Ownership']
le_map   = {}
for col in cat_cols:
    df[col] = df[col].fillna('Unknown')
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col])
    le_map[col] = le

# --- 3g. Encode location --------------------------------------
df['location'] = df['location'].fillna('Unknown')
le_loc = LabelEncoder()
df['location_enc'] = le_loc.fit_transform(df['location'])

# --- 3h. Parse Bathroom / Balcony as numeric ------------------
df['Bathroom'] = pd.to_numeric(df['Bathroom'], errors='coerce')
df['Balcony']  = pd.to_numeric(df['Balcony'],  errors='coerce')

# --- 3i. Select & clean final feature set ---------------------
features = [
    'bhk', 'area_sqft', 'floor_num', 'Bathroom', 'Balcony',
    'Furnishing_enc', 'Status_enc', 'Transaction_enc',
    'facing_enc', 'Ownership_enc', 'location_enc'
]
target = 'price_inr'

df_model = df[features + [target]].dropna()

print(f'Rows after cleaning: {len(df_model)}')
print(f'\nNull counts in cleaned dataset:')
print(df_model.isnull().sum())
display(df_model.describe())

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 4 – EXPLORATORY DATA ANALYSIS
# ─────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Exploratory Data Analysis – House Price Dataset', fontsize=14, fontweight='bold')

# Price distribution
axes[0, 0].hist(df_model['price_inr'] / 1e7, bins=40, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Price Distribution (Crores INR)')
axes[0, 0].set_xlabel('Price (Cr)')
axes[0, 0].set_ylabel('Frequency')

# BHK vs Price
bhk_price = df_model.groupby('bhk')['price_inr'].median() / 1e7
bhk_price.plot(kind='bar', ax=axes[0, 1], color='coral', edgecolor='white')
axes[0, 1].set_title('Median Price by BHK')
axes[0, 1].set_xlabel('BHK')
axes[0, 1].set_ylabel('Median Price (Cr)')
axes[0, 1].tick_params(axis='x', rotation=0)

# Area vs Price scatter
axes[0, 2].scatter(df_model['area_sqft'], df_model['price_inr'] / 1e7, alpha=0.4, s=15, color='teal')
axes[0, 2].set_title('Area vs Price')
axes[0, 2].set_xlabel('Area (sqft)')
axes[0, 2].set_ylabel('Price (Cr)')

# Furnishing vs Price
furn_price = df_model.groupby('Furnishing_enc')['price_inr'].median() / 1e7
furn_price.plot(kind='bar', ax=axes[1, 0], color='mediumpurple', edgecolor='white')
axes[1, 0].set_title('Median Price by Furnishing')
axes[1, 0].set_xlabel('Furnishing Encoded')
axes[1, 0].set_ylabel('Median Price (Cr)')
axes[1, 0].tick_params(axis='x', rotation=0)

# Correlation heatmap
corr = df_model[features + [target]].corr()
sns.heatmap(corr[['price_inr']].drop('price_inr').sort_values('price_inr'),
            annot=True, fmt='.2f', cmap='coolwarm',
            ax=axes[1, 1], cbar=True)
axes[1, 1].set_title('Feature Correlation with Price')

# Floor vs Price
floor_price = df_model.groupby('floor_num')['price_inr'].median() / 1e7
axes[1, 2].plot(floor_price.index, floor_price.values, marker='o', color='darkorange', markersize=4)
axes[1, 2].set_title('Floor Level vs Median Price')
axes[1, 2].set_xlabel('Floor')
axes[1, 2].set_ylabel('Median Price (Cr)')

plt.tight_layout()
plt.savefig('eda_charts.png', dpi=120, bbox_inches='tight')
plt.show()
print('EDA charts saved as eda_charts.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 5 – MODEL TRAINING
# ─────────────────────────────────────────────────────────────────

X = df_model[features].values
y = df_model[target].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    'Linear Regression'         : LinearRegression(),
    'Random Forest'             : RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    'Gradient Boosting'         : GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'model' : model,
        'MAE'   : mean_absolute_error(y_test, y_pred),
        'RMSE'  : np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2'    : r2_score(y_test, y_pred),
        'y_pred': y_pred
    }
    print(f'{name:30s}  MAE={results[name]["MAE"]/1e5:.2f} L  RMSE={results[name]["RMSE"]/1e5:.2f} L  R²={results[name]["R2"]:.4f}')

# Pick best model by R²
best_name  = max(results, key=lambda k: results[k]['R2'])
best_model = results[best_name]['model']
print(f'\n✅ Best model: {best_name}  (R² = {results[best_name]["R2"]:.4f})')

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 6 – MODEL EVALUATION CHARTS
# ─────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Model Evaluation', fontsize=14, fontweight='bold')

model_names  = list(results.keys())
r2_scores    = [results[n]['R2']   for n in model_names]
mae_scores   = [results[n]['MAE']  / 1e5 for n in model_names]
rmse_scores  = [results[n]['RMSE'] / 1e5 for n in model_names]

bars1 = axes[0].bar(model_names, r2_scores, color=['steelblue', 'coral', 'mediumseagreen'])
axes[0].set_title('R² Score (higher is better)')
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('R² Score')
axes[0].tick_params(axis='x', rotation=15)
for bar, val in zip(bars1, r2_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)

bars2 = axes[1].bar(model_names, mae_scores, color=['steelblue', 'coral', 'mediumseagreen'])
axes[1].set_title('MAE (lower is better)')
axes[1].set_ylabel('MAE (Lakh INR)')
axes[1].tick_params(axis='x', rotation=15)
for bar, val in zip(bars2, mae_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=9)

# Actual vs Predicted for best model
y_pred_best = results[best_name]['y_pred']
axes[2].scatter(y_test / 1e7, y_pred_best / 1e7, alpha=0.4, s=15, color='teal')
max_val = max(y_test.max(), y_pred_best.max()) / 1e7
axes[2].plot([0, max_val], [0, max_val], 'r--', lw=1.5)
axes[2].set_title(f'Actual vs Predicted – {best_name}')
axes[2].set_xlabel('Actual Price (Cr)')
axes[2].set_ylabel('Predicted Price (Cr)')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()
print('Evaluation chart saved as model_evaluation.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 7 – FEATURE IMPORTANCE (Random Forest / Gradient Boosting)
# ─────────────────────────────────────────────────────────────────

tree_models = ['Random Forest', 'Gradient Boosting']
for model_name in tree_models:
    if model_name in results:
        importances = results[model_name]['model'].feature_importances_
        sorted_idx  = np.argsort(importances)[::-1]
        plt.figure(figsize=(10, 4))
        plt.bar(np.array(features)[sorted_idx], importances[sorted_idx], color='steelblue')
        plt.title(f'Feature Importances – {model_name}')
        plt.xticks(rotation=30, ha='right')
        plt.ylabel('Importance')
        plt.tight_layout()
        plt.savefig(f'feature_importance_{model_name.replace(" ","_").lower()}.png', dpi=120, bbox_inches='tight')
        plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  CELL 8 – FRONTEND: INTERACTIVE PRICE PREDICTION UI
# ─────────────────────────────────────────────────────────────────

# ── helper: encode a free-text furnishing choice
def encode_cat(le, value):
    classes = list(le.classes_)
    return le.transform([value])[0] if value in classes else le.transform(['Unknown'])[0]

# ── get unique values for dropdowns
furnishing_opts = sorted(df['Furnishing'].dropna().unique().tolist())
status_opts     = sorted(df['Status'].dropna().unique().tolist())
txn_opts        = sorted(df['Transaction'].dropna().unique().tolist())
facing_opts     = sorted(df['facing'].dropna().unique().tolist())
ownership_opts  = sorted(df['Ownership'].dropna().unique().tolist())
location_opts   = sorted(df['location'].dropna().unique().tolist())

# ── widgets
style   = {'description_width': '160px'}
layout  = widgets.Layout(width='420px')

w_bhk         = widgets.IntSlider(value=2, min=1, max=6, step=1, description='BHK:', style=style, layout=layout)
w_area        = widgets.FloatSlider(value=600, min=100, max=5000, step=10, description='Area (sqft):', style=style, layout=layout)
w_floor       = widgets.IntSlider(value=3, min=0, max=60, step=1, description='Floor No.:', style=style, layout=layout)
w_bath        = widgets.IntSlider(value=2, min=1, max=8, step=1, description='Bathrooms:', style=style, layout=layout)
w_balcony     = widgets.IntSlider(value=1, min=0, max=4, step=1, description='Balconies:', style=style, layout=layout)
w_furnishing  = widgets.Dropdown(options=furnishing_opts, value=furnishing_opts[0], description='Furnishing:', style=style, layout=layout)
w_status      = widgets.Dropdown(options=status_opts, value=status_opts[0], description='Status:', style=style, layout=layout)
w_txn         = widgets.Dropdown(options=txn_opts, value=txn_opts[0], description='Transaction:', style=style, layout=layout)
w_facing      = widgets.Dropdown(options=['Unknown'] + facing_opts, value='Unknown', description='Facing:', style=style, layout=layout)
w_ownership   = widgets.Dropdown(options=ownership_opts, value=ownership_opts[0], description='Ownership:', style=style, layout=layout)
w_location    = widgets.Dropdown(options=location_opts, value=location_opts[0], description='Location:', style=style, layout=layout)
w_model_sel   = widgets.Dropdown(options=list(models.keys()), value=best_name, description='ML Model:', style=style, layout=layout)

btn_predict   = widgets.Button(description='Predict Price', button_style='success',
                               layout=widgets.Layout(width='180px', height='36px'))
out           = widgets.Output()

def on_predict(b):
    with out:
        clear_output(wait=True)
        feat_vec = np.array([[
            w_bhk.value,
            w_area.value,
            w_floor.value,
            w_bath.value,
            w_balcony.value,
            encode_cat(le_map['Furnishing'], w_furnishing.value),
            encode_cat(le_map['Status'],     w_status.value),
            encode_cat(le_map['Transaction'],w_txn.value),
            encode_cat(le_map['facing'],     w_facing.value),
            encode_cat(le_map['Ownership'],  w_ownership.value),
            encode_cat(le_loc,               w_location.value)
        ]])
        selected_model = results[w_model_sel.value]['model']
        pred_inr       = selected_model.predict(feat_vec)[0]

        if pred_inr >= 1e7:
            pred_str = f'₹ {pred_inr / 1e7:.2f} Crore'
        else:
            pred_str = f'₹ {pred_inr / 1e5:.2f} Lakh'

        r2_val = results[w_model_sel.value]['R2']

        display(HTML(f'''
        <div style="font-family:Segoe UI,sans-serif;border:2px solid #3b82d4;border-radius:10px;
                    padding:20px 30px;max-width:460px;background:#f0f7ff;margin-top:8px">
          <h3 style="margin:0 0 8px;color:#1d4ed8">🏠 Estimated House Price</h3>
          <p style="font-size:28px;font-weight:bold;color:#1e3a8a;margin:4px 0">{pred_str}</p>
          <hr style="border:none;border-top:1px solid #bfdbfe;margin:10px 0">
          <table style="width:100%;font-size:13px;color:#374151">
            <tr><td>Model Used</td><td><b>{w_model_sel.value}</b></td></tr>
            <tr><td>Model R² Score</td><td><b>{r2_val:.4f}</b></td></tr>
            <tr><td>BHK / Area</td><td><b>{w_bhk.value} BHK / {w_area.value:.0f} sqft</b></td></tr>
            <tr><td>Location</td><td><b>{w_location.value}</b></td></tr>
            <tr><td>Furnishing</td><td><b>{w_furnishing.value}</b></td></tr>
          </table>
        </div>
        '''))

btn_predict.on_click(on_predict)

# ── Layout
header = widgets.HTML('''
<div style="font-family:Segoe UI,sans-serif;background:#1e3a8a;color:white;
            padding:16px 24px;border-radius:8px 8px 0 0;margin-bottom:4px">
  <h2 style="margin:0">🏡 House Price Prediction</h2>
  <p style="margin:4px 0 0;font-size:13px;opacity:0.85">Powered by Machine Learning — Thane Property Dataset</p>
</div>
''')

col1 = widgets.VBox([w_bhk, w_area, w_floor, w_bath, w_balcony, w_furnishing])
col2 = widgets.VBox([w_status, w_txn, w_facing, w_ownership, w_location, w_model_sel])
cols = widgets.HBox([col1, col2], layout=widgets.Layout(gap='20px'))

ui = widgets.VBox([header, cols, btn_predict, out])
display(ui)